# 23c · Double DQN on the real factors, a decision every 60 minutes (E23 step 4)

E23a: with a decision every **60 bars** and the 8 factors, Double DQN learned a planted signal at IC 0.1 on both
seeds (validation Sharpe 1.35 and 0.70 vs buy and hold 0.52), where every 5-bar setup failed. The honest factors sit
near IC 0.1-0.13, so this is the first setup in which they may be usable. This notebook trains on them.

* **Factors (two sets):** *current* = E20a cross-fitted train factors + E15 validation factors;
  *enriched* = E23b (E15 inputs + 32 extra inputs), cross-fitted train + E23b validation factors.
* **Agent:** Double DQN, state A, E16 hyperparameters, decision every 60 bars, 3 seeds x 1M steps.
* **Honest checkpoint choice:** training is unstable (E23a checkpoints swung between Sharpe -0.75 and 4.7), so
  validation is cut in two: every checkpoint is scored on **V1** (first half, Feb - mid-Apr 2025), the best one with
  at least 5 trades is kept, and it is judged **once on V2** (mid-Apr - Jun 2025). As in E16, a run whose kept
  checkpoint does not beat cash and buy and hold on V1 trades as cash.
* **References on V2:** buy and hold, the best threshold rule chosen on train, and the best rule chosen on V1
  (the same choice the agent gets).

**Test and stress_bear are not loaded.** T4 GPU, 6 runs of about 15 min (about 1.5 h); runs resume after a
disconnect. Writes `thesis_rl/initiation/e23c_real_g60/` and `report/e23c_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "23c_real_factors_60min")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, halves as HV, walkforward as WF
from rltrade.env import EnvConfig, Market

INIT = L.root / "initiation"
OUT, REP = INIT / "e23c_real_g60", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
FACTORS = {"current": INIT / "e20" / "crossfit_mean3", "enriched": INIT / "e23_factors" / "crossfit_mean3"}
for k, d in FACTORS.items():
    assert (d / "factors_train.npy").exists() and (d / "factors_val.npy").exists(), f"{k}: missing {d} (run 20a / 23b)"
DATA = {k: HV.write_halves(PREP, d, LOCAL / "e23c" / k)[0] for k, d in FACTORS.items()}
CUTS = json.loads((DATA["current"] / "halves.json").read_text())["cuts"]

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
AGENT_HP, ENV_OVER = hp["agent_hp"], {**hp["env"], "step_gap": 60}
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
CFG = EnvConfig(state="A", **ENV_OVER)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1, 2), 1_000_000, 100_000, 5
def run_name(fs, seed): return f"{fs}_g60_s{seed}"
print("agent:", AGENT_HP, ENV_OVER, "| device", DEV, "| V1 / V2 rows:", CUTS["v1"]["rows"], CUTS["v2"]["rows"])

## 1 · References: factor IC, buy and hold, and threshold rules (chosen on train, and chosen on V1)

In [ ]:
def fwd_ic(f, price, j): return FR.ic(f[:, j].astype(np.float64), FR.forward_returns(price, FR.OFFSETS[j]), FR.OFFSETS[j])[0]
ic_rows, rule_rows = [], []
BH = {s: m["sharpe"] for s, m in HV.reference(DATA["current"], CFG, DEV).items()}
BH_RET = {s: m["cum_return_pct"] for s, m in HV.reference(DATA["current"], CFG, DEV).items()}
for fs, d in DATA.items():
    fac = {s: np.load(d / f"factors_{s}.npy") for s in HV.SPLITS}
    px = {s: np.load(d / f"market_{s}.npz")["midpoint"].astype(np.float64) for s in HV.SPLITS}
    ic_rows.append({"factors": fs, **{f"{s} f{j}": fwd_ic(fac[s], px[s], j) for s in ("train", "val") for j in (3, 5, 6, 7)}})
    grid = FR.make_grid(FR.signals(fac["train"]))
    res = {s: FR.run_rules(grid, Market(d, d, s, CFG, DEV), CFG) for s in ("train", "v1", "v2")}
    for chosen_on in ("train", "v1"):
        i = WF.select_rule(res[chosen_on], MIN_TRADES); g = grid[i]
        rule_rows.append({"factors": fs, "rule chosen on": chosen_on, "rule": f"{g['signal']} q{g['q']} {g['mode']} hold>={g['min_hold']}",
                          **{f"{s} Sharpe": res[s]["sharpe"][i] for s in ("train", "v1", "v2")},
                          "V2 return %": res["v2"]["cum_return_pct"][i], "V2 max DD %": res["v2"]["max_drawdown_pct"][i],
                          "V2 trades": int(res["v2"]["num_trades"][i])})
ic_df, rule_df = pd.DataFrame(ic_rows), pd.DataFrame(rule_rows)
rule_df.to_csv(OUT / "rule_reference.csv", index=False)
print("factor IC vs the forward return over each factor's horizon:\n" + ic_df.round(3).to_string(index=False))
print("\nbuy and hold (exact, 60-bar decisions): " + ", ".join(f"{s} Sharpe {BH[s]:+.2f} ({BH_RET[s]:+.1f} %)" for s in HV.SPLITS))
print("\nthreshold rules (decision every 60 bars):\n" + rule_df.round(3).to_string(index=False))

## 2 · Train (about 15 min per run; resumes)

In [ ]:
t0 = time.time()
for seed in SEEDS:                      # both factor sets for seed 0 first: a partial run is already informative
    for fs, d in DATA.items():
        HV.train_select(run_name(fs, seed), d, OUT, seed, AGENT_HP, ENV_OVER, TOTAL_STEPS, EVAL_EVERY, device=DEV,
                        min_trades=MIN_TRADES)
print(f"training done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results: the V1-chosen checkpoint judged on V2

In [ ]:
RES = {(fs, s): json.loads((OUT / "runs" / run_name(fs, s) / "result.json").read_text()) for fs in DATA for s in SEEDS}
rows = []
for (fs, s), r in RES.items():
    sel = r["selected"]
    row = {"factors": fs, "seed": s, "final V2 Sharpe": r["final"]["v2"]["sharpe"], "final V2 return %": r["final"]["v2"]["cum_return_pct"]}
    if sel:
        v1, v2 = sel["v1"], sel["v2"]
        traded = {"V2 Sharpe": 0.0, "V2 return %": 0.0, "V2 max DD %": 0.0, "V2 trades": 0} if sel["fallback"] else \
                 {"V2 Sharpe": v2["sharpe"], "V2 return %": v2["cum_return_pct"], "V2 max DD %": v2["max_drawdown_pct"], "V2 trades": v2["num_trades"]}
        row.update({"chosen step": sel["steps"], "V1 Sharpe (choice)": v1["sharpe"], "ckpt V2 Sharpe": v2["sharpe"],
                    "ckpt V2 return %": v2["cum_return_pct"], "ckpt V2 timing %": v2["attr_timing_pct"],
                    "ckpt V2 in market %": v2["exposure_pct"], "stayed flat": sel["fallback"], **traded})
    else:
        row.update({"chosen step": None, "stayed flat": True, "V2 Sharpe": 0.0, "V2 return %": 0.0, "V2 max DD %": 0.0, "V2 trades": 0})
    rows.append(row)
runs = pd.DataFrame(rows)
runs.to_csv(OUT / "runs_summary.csv", index=False)
agg = runs.groupby("factors")[["V2 Sharpe", "V2 return %", "V2 max DD %", "V2 trades", "ckpt V2 Sharpe", "ckpt V2 return %",
                               "final V2 Sharpe"]].agg(["mean", "std"]).round(2)
print("every run (ckpt = the checkpoint chosen on V1, judged on V2; V2 = what is traded after the stay-flat rule):")
print(runs.round(2).to_string(index=False))
print("\nper factor set, mean and SD over seeds:\n" + agg.to_string())
print(f"\nV2 references: buy and hold Sharpe {BH['v2']:+.2f} ({BH_RET['v2']:+.1f} %); rules:")
print(rule_df[["factors", "rule chosen on", "rule", "v2 Sharpe", "V2 return %", "V2 trades"]].round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key in zip(axes, ("train_sharpe", "v1_sharpe", "v2_sharpe")):
    for (fs, s), r in RES.items():
        c = pd.DataFrame(r["curve"])
        ax.plot(c.steps / 1e6, c[key], color="C0" if fs == "current" else "C1", lw=1.2, alpha=0.8,
                label=f"{fs}, seed {s}" if key == "v2_sharpe" else None)
    split = key.split("_")[0]
    ax.axhline(BH[split], color="black", ls="--", lw=1, label="buy and hold" if key == "v2_sharpe" else None)
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title({"train": "train (chunked)", "v1": "V1 (checkpoint choice)", "v2": "V2 (judged)"}[split] + ": greedy-policy Sharpe", fontsize=10)
    ax.set_xlabel("training steps (millions)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Sharpe (net of costs)"); axes[2].legend(fontsize=7)
fig.tight_layout(); fig.savefig(REP / "e23c_curves.png", dpi=130); plt.show()

In [ ]:
(REP / "e23c_real_factors_60min.md").write_text(
    "# E23c: Double DQN on the real factors, decision every 60 bars (train + validation only)\n\n"
    f"State A, E16 hyperparameters, {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps. Checkpoint = best V1 Sharpe with >= {MIN_TRADES} V1 trades, "
    "judged once on V2; stay flat if it does not beat cash and buy and hold on V1. Test and stress_bear not loaded.\n\n"
    f"V1 rows {CUTS['v1']['rows']}, V2 rows {CUTS['v2']['rows']} of validation. Buy and hold: "
    + ", ".join(f"{s} Sharpe {BH[s]:+.2f} ({BH_RET[s]:+.1f} %)" for s in HV.SPLITS) + "\n\n"
    "## Per factor set (mean, SD over seeds)\n\n" + agg.reset_index().to_markdown(index=False)
    + "\n\n## Every run\n\n" + runs.round(3).to_markdown(index=False)
    + "\n\n## Threshold rules\n\n" + rule_df.round(3).to_markdown(index=False)
    + "\n\n## Factor IC\n\n" + ic_df.round(3).to_markdown(index=False) + "\n\nFigure: e23c_curves.png\n", encoding="utf-8")
print("wrote", REP / "e23c_real_factors_60min.md")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()